# Encounter Statistics

Statistical analysis of the Gaia DR3 asteroid close-encounter catalog.
Covers distributions of distances, velocities, epochs, and orbit classes.

**Prerequisite**: Run the full pipeline first:
```bash
docker compose run --rm pipeline python -m scripts.run_pipeline
docker compose run --rm pipeline python -m scripts.characterize_catalog
```

In [ ]:
from pathlib import Path
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.catalog.query import load_catalog, filter_encounters, top_encounters

CATALOG_PATH = Path('../data/output/encounters_characterized.parquet')
df = load_catalog(CATALOG_PATH)
print(f'Loaded {len(df):,} encounters')
print(df.schema)

## Summary statistics

In [ ]:
print(f'Total encounters:        {len(df):,}')
print(f'Gaia-observable:         {int(df["gaia_observable"].sum()):,}')
print(f'Date range:              {df["date_utc"].min()} → {df["date_utc"].max()}')
print(f'Closest approach:        {float(df["dist_au"].min()):.6f} AU')
print(f'Velocity range:          {float(df["rel_vel_km_s"].min()):.3f} – {float(df["rel_vel_km_s"].max()):.3f} km/s')
print(f'Diameter body 1 (range): {float(df["diameter_1_km"].drop_nulls().min()):.0f} – {float(df["diameter_1_km"].drop_nulls().max()):.0f} km')

## Distance distribution

In [ ]:
fig = px.histogram(
    df.to_pandas(), x='dist_au', nbins=100,
    title='Distribution of minimum separation at closest approach',
    labels={'dist_au': 'Min. separation (AU)'},
    color_discrete_sequence=['#4a90d9'],
)
fig.update_layout(bargap=0.05)
fig.show()

## Velocity vs distance

In [ ]:
fig = px.scatter(
    df.sample(min(10000, len(df))).to_pandas(),
    x='dist_au', y='rel_vel_km_s',
    color='class_1',
    opacity=0.4,
    title='Relative velocity vs minimum separation (random sample of 10k)',
    labels={'dist_au': 'Min. separation (AU)', 'rel_vel_km_s': 'Velocity (km/s)', 'class_1': 'Class'},
)
fig.show()

## Temporal distribution

In [ ]:
monthly = (
    df.with_columns(pl.col('date_utc').str.slice(0, 7).alias('month'))
    .group_by('month').agg(pl.len().alias('n'))
    .sort('month')
)
fig = px.bar(
    monthly.to_pandas(), x='month', y='n',
    title='Encounters per month during Gaia DR3 window',
    labels={'month': 'Month', 'n': 'Number of encounters'},
)
fig.show()

## Orbit class breakdown

In [ ]:
class_counts = (
    pl.concat([
        df.select(pl.col('class_1').alias('class')),
        df.select(pl.col('class_2').alias('class')),
    ])
    .group_by('class').agg(pl.len().alias('count'))
    .sort('count', descending=True)
)
print(class_counts)

fig = px.pie(class_counts.to_pandas(), values='count', names='class',
             title='Encounter body class distribution (both bodies combined)')
fig.show()

## Gaia observability

In [ ]:
obs_frac = int(df['gaia_observable'].sum()) / len(df)
print(f'Gaia-observable fraction: {obs_frac:.1%}')

fig = px.scatter(
    df.sample(min(5000, len(df))).to_pandas(),
    x='solar_elongation_deg', y='dist_au',
    color='gaia_observable',
    opacity=0.4,
    color_discrete_map={True: '#2ecc71', False: '#e74c3c'},
    title='Solar elongation vs encounter distance (Gaia observability)',
    labels={'solar_elongation_deg': 'Solar elongation (deg)', 'dist_au': 'Min. separation (AU)'},
)
fig.add_vline(x=45, line_dash='dash', annotation_text='Gaia exclusion zone')
fig.show()